# synapse-sr quick start: Sentinel-2 to 2 m

Runs on **Google Colab** and **Kaggle** (and any Jupyter). A GPU is strongly recommended (the free CPU runtimes have only 2 cores): *Runtime → Change runtime type → GPU*
on Colab, *Settings → Accelerator → GPU* on Kaggle. On Kaggle, also switch *Internet* on.

What this notebook does:

1. installs `synapse-sr` and checks the environment;
2. downloads a real Sentinel-2 L2A scene (Bengaluru, cloud-free) from a public catalogue;
3. super-resolves it to 2 m and shows what the satellite determined versus what the model added;
4. uses the result for **crop monitoring**, **urban analysis** and **flood / disaster change detection**.

In [ ]:
%pip install -q "synapse-sr[stac,xarray]" matplotlib

In [ ]:
!synapse-sr --env

`pro_scan_backend` should read `triton` or `fused` on a GPU runtime and `pytorch` on CPU. All of them give the same
numbers; only the speed differs.

## 1. Get a Sentinel-2 scene
`fetch_sentinel2` picks the least-cloudy L2A scene over a point and writes a ready-to-use GeoTIFF, plus its
scene-classification cloud mask (applied automatically).

In [ ]:
import torch
import synapse_sr
from synapse_sr import Pro, Flash

GPU = torch.cuda.is_available()
SIZE_M = 1280 if GPU else 480        # Pro on a 2-core CPU runtime is slow: keep the area small without a GPU
print("GPU runtime" if GPU else "CPU runtime: using a 480 m area (switch on a GPU for full-size scenes)")

scene = synapse_sr.fetch_sentinel2(lat=12.9237, lon=77.4987, start="2025-01-01", end="2025-03-15", size_m=SIZE_M)
scene

## 2. Super-resolve
The first call downloads the weights once (~58 MB, SHA-256 verified) and shows a progress bar.

In [ ]:
model = Pro.from_pretrained()          # or Flash.from_pretrained() for CPU-only machines
r = model.super_resolve(scene)
r.summary();

In [ ]:
r.show(["image", "x_base", "prior", "support"]);

* **image**: the 2 m result.
* **x_base**: what the 10 m measurement alone determines (a physics-only inversion).
* **prior**: the detail the network added. It is invisible to the sensor by construction, so the satellite data
  can neither confirm nor contradict it.
* **support**: green = observation-determined, amber = medium, red = prior-dominated or masked.

Save a georeferenced GeoTIFF (4 reflectance bands + error scale + support):

In [ ]:
r.save("bengaluru_2m.tif")

## 3. Crop monitoring
Vegetation indices at 2 m, field-boundary strength, and statistics restricted to trustworthy pixels.

In [ ]:
import matplotlib.pyplot as plt
idx = r.indices()                                   # ndvi savi evi gndvi ndwi ndre ndbi nbr mndwi
fields = synapse_sr.boundaries(r, "field")
trusted = r.support == 2
print("mean NDVI (observation-determined pixels):", float(idx["ndvi"][trusted].mean()))
fig, ax = plt.subplots(1, 2, figsize=(10, 5))
ax[0].imshow(idx["ndvi"], cmap="RdYlGn", vmin=-0.2, vmax=0.9); ax[0].set_title("NDVI, 2 m")
ax[1].imshow(fields, cmap="gray_r"); ax[1].set_title("field boundaries")
[a.set_axis_off() for a in ax];

## 4. Urban analysis

In [ ]:
urban = synapse_sr.boundaries(r, "urban")           # building and road edges
fig, ax = plt.subplots(1, 2, figsize=(10, 5))
ax[0].imshow(r.rgb()); ax[0].set_title("2 m true colour")
ax[1].imshow(urban, cmap="magma"); ax[1].set_title("urban edges")
[a.set_axis_off() for a in ax];

## 5. Flood and disaster change detection
Two dates over the same area. `change` flags a pixel only when it is valid and observation-supported on **both**
dates, and reports how much of the area could not be assessed.

The example below uses Kerala before and after the 2024 monsoon; replace the coordinates and dates with your event.

In [ ]:
kw = dict(lat=11.4686, lon=76.1350, size_m=SIZE_M)                 # Wayanad, Kerala
before = model.super_resolve(synapse_sr.fetch_sentinel2(start="2024-02-01", end="2024-04-30", **kw))
after = model.super_resolve(synapse_sr.fetch_sentinel2(start="2024-10-01", end="2024-12-31", **kw))

veg = synapse_sr.change(before, after, "ndvi")                    # vegetation loss: landslide scars
print(f"changed area {veg.area_km2:.3f} km2, not assessable {100 * veg.unreliable_fraction:.0f}%")
fig, ax = plt.subplots(1, 3, figsize=(14, 5))
ax[0].imshow(before.rgb()); ax[0].set_title("before")
ax[1].imshow(after.rgb()); ax[1].set_title("after")
ax[2].imshow(veg.delta, cmap="RdBu", vmin=-0.5, vmax=0.5); ax[2].contour(veg.mask, colors="k", linewidths=0.5)
ax[2].set_title("NDVI change (contours: flagged)")
[a.set_axis_off() for a in ax];

Other change indices: `"ndwi"` (flood water), `"nbr"` (burn scars), `"ndbi"` (construction), `"brightness"`
(debris, bare soil).

## 6. Your own data
```python
r = model.super_resolve("my_scene.tif")                          # GeoTIFF with 10, 12 or 13 bands or named bands
r = model.super_resolve(array, band_names=["B04", "B03", ...])   # numpy / torch (C, H, W)
r = model.super_resolve(cube.isel(time=0))                       # xarray (cubo, stackstac)
```
Full documentation: https://sharadhnaidu.github.io/synapse-sr/